In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import random
import json
from datetime import datetime

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader
from datasets import Dataset
from transformers import AutoTokenizer, AutoModel, DataCollatorWithPadding
from tqdm.auto import tqdm

In [2]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

PyTorch: 2.5.1+cu121
CUDA available: True
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
CUDA: 12.1


device(type='cuda')

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [4]:
# ============================================================
# 3. LOAD GENDER PERSPECTIVE DATASET
# ============================================================

DATA_PATH = "../data/processed/gender_perspective_hatespeech_softlabels_min2.csv"

gender_df = pd.read_csv(DATA_PATH)

print("Gender perspective dataset:", gender_df.shape)
print("Unique comments:", gender_df["comment_id"].nunique())
print("Perspectives:", gender_df["perspective"].unique())

gender_df.head()

Gender perspective dataset: (8089, 12)
Unique comments: 7582
Perspectives: ['women' 'men' 'non_binary' 'prefer_not_to_say']


,comment_id,perspective,hatespeech_0_prob,hatespeech_1_prob,hatespeech_2_prob,annotation_count,split,text_clean,perspective_type,input_text,entropy,distribution_pattern
0,10,women,0.333333,0.0,0.666667,3,train,"They'll come back in your plan, also. Plus we ...",annotator_gender_group,"[WOMEN] They'll come back in your plan, also. ...",0.918296,"(0.333, 0.0, 0.667)"
1,12,women,0.500000,0.0,0.500000,2,train,She ugly anyways,annotator_gender_group,[WOMEN] She ugly anyways,1.000000,"(0.5, 0.0, 0.5)"
2,19,men,1.000000,0.0,0.000000,2,train,Won't happen. Birth rates are down. Women want...,annotator_gender_group,[MEN] Won't happen. Birth rates are down. Wome...,-0.000000,"(1.0, 0.0, 0.0)"
3,22,men,0.500000,0.0,0.500000,2,train,The guys are one thing but then you have a wom...,annotator_gender_group,[MEN] The guys are one thing but then you have...,1.000000,"(0.5, 0.0, 0.5)"
4,26,men,0.500000,0.0,0.500000,2,train,I'd love to rip those panties off and shove my...,annotator_gender_group,[MEN] I'd love to rip those panties off and sh...,1.000000,"(0.5, 0.0, 0.5)"


In [5]:
# ============================================================
# 4. CHECK REQUIRED COLUMNS
# ============================================================

required_cols = [
    "comment_id",
    "perspective",
    "input_text",
    "split",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "annotation_count",
    "entropy"
]

missing_cols = [col for col in required_cols if col not in gender_df.columns]

print("Missing columns:", missing_cols)

if len(missing_cols) == 0:
    print("All required columns are present.")

Missing columns: []
All required columns are present.


In [6]:
# ============================================================
# 5. LABEL COLUMNS AND PERSPECTIVES
# ============================================================

LABELS = [0, 1, 2]

label_cols = [
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob"
]

ALLOWED_GENDER_PERSPECTIVES = [
    "women",
    "men",
    "non_binary",
    "prefer_not_to_say"
]

gender_df = gender_df[
    gender_df["perspective"].isin(ALLOWED_GENDER_PERSPECTIVES)
].copy()

print(gender_df["perspective"].value_counts())

perspective
women                4850
men                  3200
non_binary             28
prefer_not_to_say      11
Name: count, dtype: int64


In [7]:
# ============================================================
# 6. DATASET SUMMARY
# ============================================================

perspective_summary = (
    gender_df
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2)),
        nonzero_entropy_percent=("entropy", lambda x: round((x > 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

display(perspective_summary)

split_summary = (
    gender_df
    .groupby("split")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2)),
        unique_perspectives=("perspective", "nunique")
    )
    .reset_index()
)

display(split_summary)

perspective_split_summary = (
    gender_df
    .groupby(["split", "perspective"])
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotation_count=("annotation_count", "mean"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values(["split", "examples"], ascending=[True, False])
)

display(perspective_split_summary)

,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent,nonzero_entropy_percent
3,women,4850,4850,3.785361,2.0,0.381104,61.57,38.43
0,men,3200,3200,3.879688,2.0,0.362966,63.22,36.78
1,non_binary,28,28,3.607143,3.0,0.481488,46.43,53.57
2,prefer_not_to_say,11,11,3.181818,2.0,0.306450,72.73,27.27


,split,examples,unique_comments,mean_annotation_count,mean_entropy,zero_entropy_percent,unique_perspectives
0,test,1224,1140,4.035131,0.376000,62.09,4
1,train,5672,5316,3.783850,0.374394,62.17,4
2,validation,1193,1126,3.779547,0.371255,62.36,3


,split,perspective,examples,unique_comments,mean_annotation_count,mean_entropy,zero_entropy_percent
3,test,women,710,710,4.059155,0.376094,62.25
0,test,men,507,507,3.998028,0.377073,61.74
1,test,non_binary,4,4,5.250000,0.162506,75.00
2,test,prefer_not_to_say,3,3,3.000000,0.456984,66.67
7,train,women,3392,3392,3.757075,0.379244,61.73
4,train,men,2251,2251,3.831186,0.366411,62.91
5,train,non_binary,21,21,3.238095,0.494293,47.62
6,train,prefer_not_to_say,8,8,3.250000,0.250000,75.00
10,validation,women,748,748,3.653743,0.394293,60.16
8,validation,men,442,442,3.990950,0.329240,66.52


In [8]:
gender_df = gender_df[
    (gender_df["annotation_count"] >= 2) &
    (gender_df["entropy"] > 0)
].copy()

print("After removing zero-entropy examples:", gender_df.shape)
print(gender_df["perspective"].value_counts())

After removing zero-entropy examples: (3059, 12)
perspective
women                1864
men                  1177
non_binary             15
prefer_not_to_say       3
Name: count, dtype: int64


In [9]:

train_df = gender_df[gender_df["split"] == "train"].copy()
val_df = gender_df[gender_df["split"] == "validation"].copy()
test_df = gender_df[gender_df["split"] == "test"].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain perspectives:")
print(train_df["perspective"].value_counts())

print("\nValidation perspectives:")
print(val_df["perspective"].value_counts())

print("\nTest perspectives:")
print(test_df["perspective"].value_counts())

Train: (2146, 12)
Validation: (449, 12)
Test: (464, 12)

Train perspectives:
perspective
women                1298
men                   835
non_binary             11
prefer_not_to_say       2
Name: count, dtype: int64

Validation perspectives:
perspective
women         298
men           148
non_binary      3
Name: count, dtype: int64

Test perspectives:
perspective
women                268
men                  194
non_binary             1
prefer_not_to_say      1
Name: count, dtype: int64


In [10]:
hf_cols = [
    "comment_id",
    "perspective",
    "input_text",
    "hatespeech_0_prob",
    "hatespeech_1_prob",
    "hatespeech_2_prob",
    "annotation_count",
    "entropy"
]

train_dataset = Dataset.from_pandas(train_df[hf_cols].reset_index(drop=True))
val_dataset = Dataset.from_pandas(val_df[hf_cols].reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df[hf_cols].reset_index(drop=True))

In [11]:
model_name = "roberta-base"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(model_name)

special_tokens = {
    "additional_special_tokens": [
        "[WOMEN]",
        "[MEN]",
        "[NON_BINARY]",
        "[PREFER_NOT_TO_SAY]"
    ]
}

num_added = tokenizer.add_special_tokens(special_tokens)

print("Added special tokens:", num_added)
print("Tokenizer size:", len(tokenizer))

Added special tokens: 4
Tokenizer size: 50269


In [12]:
def tokenize(batch):
    return tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=MAX_LENGTH
    )

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/2146 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/464 [00:00<?, ? examples/s]

In [13]:
def add_labels(batch):
    n = len(batch["input_text"])

    batch["labels"] = [
        [
            float(batch["hatespeech_0_prob"][i]),
            float(batch["hatespeech_1_prob"][i]),
            float(batch["hatespeech_2_prob"][i])
        ]
        for i in range(n)
    ]

    return batch

train_dataset = train_dataset.map(add_labels, batched=True)
val_dataset = val_dataset.map(add_labels, batched=True)
test_dataset = test_dataset.map(add_labels, batched=True)

Map:   0%|          | 0/2146 [00:00<?, ? examples/s]

Map:   0%|          | 0/449 [00:00<?, ? examples/s]

Map:   0%|          | 0/464 [00:00<?, ? examples/s]

In [14]:
columns_to_keep = [
    "input_ids",
    "attention_mask",
    "labels"
]

train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

In [24]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

test_loader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

In [15]:
class GenderPerspectiveLeWiDiModel(nn.Module):
    def __init__(self, model_name, tokenizer_size, dropout_prob=0.35, freeze_layers=4):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        self.encoder.resize_token_embeddings(tokenizer_size)

        hidden_size = self.encoder.config.hidden_size

        if freeze_layers > 0:
            for param in self.encoder.embeddings.parameters():
                param.requires_grad = False

            for layer in self.encoder.encoder.layer[:freeze_layers]:
                for param in layer.parameters():
                    param.requires_grad = False

        self.dropout = nn.Dropout(dropout_prob)
        self.classifier = nn.Linear(hidden_size, 3)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        pooled = outputs.last_hidden_state[:, 0, :]
        pooled = self.dropout(pooled)

        logits = self.classifier(pooled)
        return logits

In [16]:
model = GenderPerspectiveLeWiDiModel(
    model_name=model_name,
    tokenizer_size=len(tokenizer),
    dropout_prob=0.35,
    freeze_layers=4
).to(device)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)
print("Trainable percent:", round(trainable_params / total_params * 100, 2))

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainable params: 57295875
Total params: 124651011
Trainable percent: 45.97


In [17]:
def compute_loss(model, batch):
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels = batch["labels"].float().to(device)

    logits = model(
        input_ids=input_ids,
        attention_mask=attention_mask
    )

    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    loss = torch.nn.functional.kl_div(
        log_probs,
        labels,
        reduction="batchmean"
    )

    return loss

In [18]:
def distribution_metrics(probs, labels):
    eps = 1e-12

    kl = np.sum(
        labels * (np.log(labels + eps) - np.log(probs + eps)),
        axis=1
    ).mean()

    m = 0.5 * (labels + probs)

    js = 0.5 * (
        np.sum(labels * (np.log(labels + eps) - np.log(m + eps)), axis=1)
        +
        np.sum(probs * (np.log(probs + eps) - np.log(m + eps)), axis=1)
    ).mean()

    hard_accuracy = (probs.argmax(axis=1) == labels.argmax(axis=1)).mean()

    pred_entropy = -np.sum(probs * np.log2(probs + eps), axis=1)
    gold_entropy = -np.sum(labels * np.log2(labels + eps), axis=1)

    if np.std(pred_entropy) == 0 or np.std(gold_entropy) == 0:
        entropy_corr = np.nan
    else:
        entropy_corr = np.corrcoef(pred_entropy, gold_entropy)[0, 1]

    entropy_mae = np.mean(np.abs(pred_entropy - gold_entropy))

    return {
        "kl_divergence": float(kl),
        "js_divergence": float(js),
        "hard_accuracy": float(hard_accuracy),
        "entropy_correlation": float(entropy_corr),
        "entropy_mae": float(entropy_mae),
    }

In [19]:
def predict_loader(model, loader):
    model.eval()

    all_probs = []
    all_labels = []
    losses = []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            labels = batch["labels"].float()
            loss = compute_loss(model, batch)

            logits = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )

            probs = torch.nn.functional.softmax(logits, dim=-1)

            losses.append(loss.item())
            all_probs.append(probs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    probs = np.vstack(all_probs)
    labels = np.vstack(all_labels)

    return probs, labels, float(np.mean(losses))

In [20]:
def evaluate_overall(model, loader):
    probs, labels, loss = predict_loader(model, loader)
    metrics = distribution_metrics(probs, labels)
    metrics["eval_loss"] = loss
    return metrics

In [21]:
def evaluate_by_perspective(model, loader, original_df):
    probs, labels, _ = predict_loader(model, loader)

    out = original_df.reset_index(drop=True).copy()

    out["pred_0"] = probs[:, 0]
    out["pred_1"] = probs[:, 1]
    out["pred_2"] = probs[:, 2]

    out["gold_0"] = labels[:, 0]
    out["gold_1"] = labels[:, 1]
    out["gold_2"] = labels[:, 2]

    rows = []

    for perspective, group_df in out.groupby("perspective"):
        group_probs = group_df[["pred_0", "pred_1", "pred_2"]].values
        group_labels = group_df[["gold_0", "gold_1", "gold_2"]].values

        metrics = distribution_metrics(group_probs, group_labels)

        row = {
            "perspective": perspective,
            "n": len(group_df),
            "mean_annotation_count": group_df["annotation_count"].mean(),
            "mean_entropy": group_df["entropy"].mean(),
        }

        row.update(metrics)
        rows.append(row)

    return pd.DataFrame(rows).sort_values("n", ascending=False), out

In [22]:
LEARNING_RATE = 5e-6
WEIGHT_DECAY = 0.10
NUM_EPOCHS = 10
PATIENCE = 3

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [25]:
output_dir = "../models/roberta_gender_perspective_conditioned_lewidi_2"
best_model_dir = os.path.join(output_dir, "best_model")

os.makedirs(best_model_dir, exist_ok=True)

history_rows = []

best_val_js = float("inf")
epochs_without_improvement = 0

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()

    train_losses = []

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{NUM_EPOCHS}")

    for batch in progress:
        batch = {k: v.to(device) for k, v in batch.items()}

        optimizer.zero_grad()

        loss = compute_loss(model, batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.requires_grad],
            max_norm=1.0
        )

        optimizer.step()

        train_losses.append(loss.item())

        progress.set_postfix({
            "train_loss": np.mean(train_losses)
        })

    val_metrics = evaluate_overall(model, val_loader)
    val_perspective_metrics, _ = evaluate_by_perspective(
        model,
        val_loader,
        val_df
    )

    print("\n" + "=" * 80)
    print(f"Epoch {epoch}")
    print(f"Train loss: {np.mean(train_losses):.6f}")
    print("Validation overall:")
    for k, v in val_metrics.items():
        print(f"{k}: {v}")

    print("\nValidation by perspective:")
    print(val_perspective_metrics.to_string(index=False))
    print("=" * 80 + "\n")

    history_row = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
    }

    for k, v in val_metrics.items():
        history_row[f"val_{k}"] = v

    history_rows.append(history_row)

    current_val_js = val_metrics["js_divergence"]

    if current_val_js < best_val_js:
        best_val_js = current_val_js
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            os.path.join(best_model_dir, "pytorch_model.bin")
        )

        tokenizer.save_pretrained(best_model_dir)

        print(f"New best model saved. Best validation JS: {best_val_js:.6f}")

    else:
        epochs_without_improvement += 1

        print(f"No improvement. Patience: {epochs_without_improvement}/{PATIENCE}")

        if epochs_without_improvement >= PATIENCE:
            print("Early stopping triggered.")
            break

    torch.cuda.empty_cache()

Epoch 1/10:   0%|          | 0/269 [00:00<?, ?it/s]


Epoch 1
Train loss: 0.391901
Validation overall:
kl_divergence: 0.361558198928833
js_divergence: 0.11147642135620117
hard_accuracy: 0.6458797327394209
entropy_correlation: -0.028856444854200625
entropy_mae: 0.5101560950279236
eval_loss: 0.36998828012367774

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 298               5.822148      0.989701       0.370190       0.114261       0.677852            -0.068997     0.509333
        men 148               7.472973      0.983272       0.337563       0.104178       0.587838             0.035977     0.508211
 non_binary   3               4.000000      0.817167       0.687902       0.194998       0.333333             0.927249     0.687901

New best model saved. Best validation JS: 0.111476


Epoch 2/10:   0%|          | 0/269 [00:00<?, ?it/s]


Epoch 2
Train loss: 0.370625
Validation overall:
kl_divergence: 0.3590010404586792
js_divergence: 0.10968878865242004
hard_accuracy: 0.6993318485523385
entropy_correlation: 0.008479342716791178
entropy_mae: 0.49339163303375244
eval_loss: 0.36787018334043436

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 298               5.822148      0.989701       0.367499       0.112481       0.714765             0.010093     0.492557
        men 148               7.472973      0.983272       0.334823       0.102273       0.675676             0.013808     0.491437
 non_binary   3               4.000000      0.817167       0.707654       0.198161       0.333333             0.288265     0.672724

New best model saved. Best validation JS: 0.109689


Epoch 3/10:   0%|          | 0/269 [00:00<?, ?it/s]

...


Epoch 6
Train loss: 0.317803
Validation overall:
kl_divergence: 0.3762512803077698
js_divergence: 0.10684521496295929
hard_accuracy: 0.5634743875278396
entropy_correlation: 0.044737085784278936
entropy_mae: 0.4089992046356201
eval_loss: 0.3839748702172575

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 298               5.822148      0.989701       0.387342       0.110326       0.580537            -0.000261     0.414715
        men 148               7.472973      0.983272       0.347414       0.098315       0.533784             0.124847     0.394793
 non_binary   3               4.000000      0.817167       0.697215       0.181915       0.333333             0.581795     0.542077

No improvement. Patience: 1/3


Epoch 7/10:   0%|          | 0/269 [00:00<?, ?it/s]


Epoch 7
Train loss: 0.308380
Validation overall:
kl_divergence: 0.3744750916957855
js_divergence: 0.10847247391939163
hard_accuracy: 0.5946547884187082
entropy_correlation: 0.024866388560832363
entropy_mae: 0.4360654652118683
eval_loss: 0.3789226381943144

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 298               5.822148      0.989701       0.383757       0.111447       0.607383            -0.015790     0.439961
        men 148               7.472973      0.983272       0.351334       0.101341       0.574324             0.102103     0.425070
 non_binary   3               4.000000      0.817167       0.594113       0.164826       0.333333             0.264183     0.591574

No improvement. Patience: 2/3


Epoch 8/10:   0%|          | 0/269 [00:00<?, ?it/s]


Epoch 8
Train loss: 0.289561
Validation overall:
kl_divergence: 0.3873574733734131
js_divergence: 0.11004319787025452
hard_accuracy: 0.5902004454342984
entropy_correlation: 0.020029144357480034
entropy_mae: 0.4145389497280121
eval_loss: 0.39323810256760694

Validation by perspective:
perspective   n  mean_annotation_count  mean_entropy  kl_divergence  js_divergence  hard_accuracy  entropy_correlation  entropy_mae
      women 298               5.822148      0.989701       0.393861       0.112106       0.607383            -0.014374     0.417946
        men 148               7.472973      0.983272       0.370810       0.104953       0.560811             0.096380     0.404121
 non_binary   3               4.000000      0.817167       0.557629       0.156226       0.333333             0.207387     0.590075

No improvement. Patience: 3/3
Early stopping triggered.


In [26]:
model.load_state_dict(
    torch.load(
        os.path.join(best_model_dir, "pytorch_model.bin"),
        map_location=device
    )
)

model.to(device)

val_results = evaluate_overall(model, val_loader)
test_results = evaluate_overall(model, test_loader)

val_perspective_results, val_pred_df = evaluate_by_perspective(
    model,
    val_loader,
    val_df
)

test_perspective_results, test_pred_df = evaluate_by_perspective(
    model,
    test_loader,
    test_df
)

print("Final validation overall:")
print(val_results)

print("\nFinal validation by perspective:")
display(val_perspective_results)

print("\nFinal test overall:")
print(test_results)

print("\nFinal test by perspective:")
display(test_perspective_results)

/tmp/ipykernel_3257/812031077.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(


Final validation overall:
{'kl_divergence': 0.36029142141342163, 'js_divergence': 0.10629386454820633, 'hard_accuracy': 0.6169265033407573, 'entropy_correlation': 0.0755952830071677, 'entropy_mae': 0.449850469827652, 'eval_loss': 0.36839519081444577}

Final validation by perspective:


,perspective,n,mean_annotation_count,mean_entropy,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
2,women,298,5.822148,0.989701,0.370237,0.109666,0.634228,0.038629,0.455388
0,men,148,7.472973,0.983272,0.334312,0.098097,0.587838,0.133353,0.436087
1,non_binary,3,4.000000,0.817167,0.653992,0.175712,0.333333,0.430690,0.578849



Final test overall:
{'kl_divergence': 0.35329747200012207, 'js_divergence': 0.10474921762943268, 'hard_accuracy': 0.5711206896551724, 'entropy_correlation': 0.030922804493705106, 'entropy_mae': 0.4432970881462097, 'eval_loss': 0.35329746377879173}

Final test by perspective:


,perspective,n,mean_annotation_count,mean_entropy,kl_divergence,js_divergence,hard_accuracy,entropy_correlation,entropy_mae
3,women,268,6.947761,0.996367,0.352796,0.104921,0.582090,-0.020101,0.446876
0,men,194,6.603093,0.985443,0.355254,0.104842,0.551546,0.109792,0.439211
1,non_binary,1,6.000000,0.650022,0.398078,0.130015,1.000000,NaN,0.685843
2,prefer_not_to_say,1,5.000000,1.370951,0.063188,0.015474,1.000000,NaN,0.034364


In [27]:
pyevall_dir = "../results/pyevall/gender_perspective_conditioned_lewidi_2"
os.makedirs(pyevall_dir, exist_ok=True)

gold_records = []
pred_records = []

for _, row in test_pred_df.iterrows():
    record_id = str(row["comment_id"]) + "_" + str(row["perspective"])

    gold_records.append({
        "test_case": "gender_perspective_hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["gold_0"]),
            "1": float(row["gold_1"]),
            "2": float(row["gold_2"])
        }
    })

    pred_records.append({
        "test_case": "gender_perspective_hatespeech",
        "id": record_id,
        "value": {
            "0": float(row["pred_0"]),
            "1": float(row["pred_1"]),
            "2": float(row["pred_2"])
        }
    })

gold_path = os.path.join(pyevall_dir, "gender_perspective_hatespeech_gold.json")
pred_path = os.path.join(pyevall_dir, "gender_perspective_hatespeech_pred.json")

with open(gold_path, "w", encoding="utf-8") as f:
    json.dump(gold_records, f, indent=2)

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(pred_records, f, indent=2)

print("Gold:", gold_path)
print("Pred:", pred_path)

Gold: ../results/pyevall/gender_perspective_conditioned_lewidi_2/gender_perspective_hatespeech_gold.json
Pred: ../results/pyevall/gender_perspective_conditioned_lewidi_2/gender_perspective_hatespeech_pred.json


In [28]:
from pyevall.evaluation import PyEvALLEvaluation
from pyevall.utils.utils import PyEvALLUtils

params = {
    PyEvALLUtils.PARAM_REPORT: PyEvALLUtils.PARAM_OPTION_REPORT_EMBEDDED
}

evaluator = PyEvALLEvaluation()

pyevall_report = evaluator.evaluate(
    pred_path,
    gold_path,
    [
        "CrossEntropy",
        "ICMSoft",
        "ICMSoftNorm"
    ],
    **params
)

pyevall_data = pyevall_report.report if hasattr(pyevall_report, "report") else pyevall_report.__dict__
pyevall_data

2026-06-03 12:53:03,971 - pyevall.evaluation - INFO -             evaluate() - Evaluating the following metrics ['CrossEntropy', 'ICMSoft', 'ICMSoftNorm']
2026-06-03 12:53:04,067 - pyevall.metrics.metrics - INFO -             evaluate() - Executing Cross Entropy evaluation method for testcase gender_perspective_hatespeech
2026-06-03 12:53:04,169 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech
2026-06-03 12:53:05,108 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM-Soft Normalized evaluation method for testcase gender_perspective_hatespeech
2026-06-03 12:53:05,109 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech
2026-06-03 12:53:05,295 - pyevall.metrics.metrics - INFO -             evaluate() - Executing ICM Soft evaluation method for testcase gender_perspective_hatespeech
cargado 4

{'metrics': {'CrossEntropy': {'name': 'Cross Entropy',
   'acronym': 'CE',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': 1.5022944377654732}],
    'average_per_test_case': 1.5022944377654732}},
  'ICMSoft': {'name': 'Information Contrast Model Soft',
   'acronym': 'ICM-Soft',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': -1.3131203459673424}],
    'average_per_test_case': -1.3131203459673424}},
  'ICMSoftNorm': {'name': 'Normalized Information Contrast Model Soft',
   'acronym': 'ICM-Soft-Norm',
   'description': 'Coming soon!',
   'status': 'OK',
   'results': {'test_cases': [{'name': 'gender_perspective_hatespeech',
      'average': 0.3562307634549466}],
    'average_per_test_case': 0.3562307634549466}}},
 'files': {'gender_perspective_hatespeech_pred.json': {'name': 'gender_perspective_hatespee

In [29]:
pyevall_summary = {}

for metric_name, metric_data in pyevall_data["metrics"].items():
    pyevall_summary[metric_name] = metric_data["results"]["average_per_test_case"]

pyevall_summary

{'CrossEntropy': 1.5022944377654732,
 'ICMSoft': -1.3131203459673424,
 'ICMSoftNorm': 0.3562307634549466}

In [30]:
os.makedirs("../results/tables", exist_ok=True)

results_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_2_results.txt"
history_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_2_training_history.csv"
test_pred_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_2_test_predictions.csv"
test_perspective_path = "../results/tables/roberta_gender_perspective_conditioned_lewidi_2_test_perspective_metrics.csv"

history_df = pd.DataFrame(history_rows)

history_df.to_csv(history_path, index=False)
test_pred_df.to_csv(test_pred_path, index=False)
test_perspective_results.to_csv(test_perspective_path, index=False)

with open(results_path, "w", encoding="utf-8") as f:
    f.write("ROBERTA GENDER-PERSPECTIVE CONDITIONED LEWIDI MODEL RESULTS\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. RUN INFORMATION\n")
    f.write("-" * 90 + "\n")
    f.write(f"Run timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"Model name: {model_name}\n")
    f.write(f"Output directory: {output_dir}\n")
    f.write(f"Best validation JS: {best_val_js}\n")
    f.write(f"Epochs completed: {len(history_df)}\n\n")

    f.write("2. DATASET STRATEGY\n")
    f.write("-" * 90 + "\n")
    f.write("Task: perspective-conditioned hatespeech soft-label prediction.\n")
    f.write("Each example is a comment-perspective pair.\n")
    f.write("Input format: [PERSPECTIVE] comment text.\n")
    f.write("Target: perspective-specific hatespeech annotator distribution.\n")
    f.write(f"Minimum annotations per comment-perspective pair: 2\n\n")

    f.write("Gender perspectives included:\n")
    for p in ALLOWED_GENDER_PERSPECTIVES:
        f.write(f"- {p}\n")
    f.write("\n")

    f.write("Perspective summary:\n")
    f.write(perspective_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Split summary:\n")
    f.write(split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("Perspective split summary:\n")
    f.write(perspective_split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("3. MODEL OBJECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write("Architecture: shared RoBERTa encoder with one 3-class distribution head.\n")
    f.write("Perspective is provided as a special input token, not as a prediction target.\n")
    f.write("Loss: KL divergence between predicted and perspective-specific gold distributions.\n\n")

    f.write("4. TRAINING SETUP\n")
    f.write("-" * 90 + "\n")
    f.write(f"Max length: {MAX_LENGTH}\n")
    f.write(f"Train batch size: {BATCH_SIZE}\n")
    f.write(f"Eval batch size: {EVAL_BATCH_SIZE}\n")
    f.write(f"Learning rate: {LEARNING_RATE}\n")
    f.write(f"Weight decay: {WEIGHT_DECAY}\n")
    f.write("Dropout: 0.35\n")
    f.write("Frozen lower RoBERTa layers: 4\n")
    f.write(f"Max epochs: {NUM_EPOCHS}\n")
    f.write(f"Early stopping patience: {PATIENCE}\n\n")

    f.write("5. TRAINING HISTORY\n")
    f.write("-" * 90 + "\n")
    f.write(history_df.to_string(index=False))
    f.write("\n\n")

    f.write("6. FINAL VALIDATION RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in val_results.items():
        f.write(f"{k}: {v}\n")

    f.write("\nValidation by perspective:\n")
    f.write(val_perspective_results.to_string(index=False))
    f.write("\n\n")

    f.write("7. FINAL TEST RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in test_results.items():
        f.write(f"{k}: {v}\n")

    f.write("\nTest by perspective:\n")
    f.write(test_perspective_results.to_string(index=False))
    f.write("\n\n")

    f.write("8. PYEVAL LEWIDI RESULTS\n")
    f.write("-" * 90 + "\n")
    for k, v in pyevall_summary.items():
        f.write(f"{k}: {v}\n")
    f.write("\n")

    

print("Saved:", results_path)
print(open(results_path, encoding="utf-8").read())

Saved: ../results/tables/roberta_gender_perspective_conditioned_lewidi_2_results.txt
ROBERTA GENDER-PERSPECTIVE CONDITIONED LEWIDI MODEL RESULTS

1. RUN INFORMATION
------------------------------------------------------------------------------------------
Run timestamp: 2026-06-03 12:53:05
Model name: roberta-base
Output directory: ../models/roberta_gender_perspective_conditioned_lewidi_2
Best validation JS: 0.10629386454820633
Epochs completed: 8

2. DATASET STRATEGY
------------------------------------------------------------------------------------------
Task: perspective-conditioned hatespeech soft-label prediction.
Each example is a comment-perspective pair.
Input format: [PERSPECTIVE] comment text.
Target: perspective-specific hatespeech annotator distribution.
Minimum annotations per comment-perspective pair: 2

Gender perspectives included:
- women
- men
- non_binary
- prefer_not_to_say

Perspective summary:
      perspective  examples  unique_comments  mean_annotations  median

In [31]:
# ============================================================
# GENDER SUBGROUP DIVERGENCE AUDIT
# Checks whether women and men distributions actually differ
# ============================================================

import numpy as np
import pandas as pd

def js_divergence(p, q):
    eps = 1e-12

    p = np.array(p, dtype=float)
    q = np.array(q, dtype=float)

    m = 0.5 * (p + q)

    js = 0.5 * (
        np.sum(p * (np.log(p + eps) - np.log(m + eps))) +
        np.sum(q * (np.log(q + eps) - np.log(m + eps)))
    )

    return js

In [32]:
# Keep only women and men distributions
wm_df = gender_df[
    gender_df["perspective"].isin(["women", "men"])
].copy()

# Wide format: one row per comment, with women and men distributions side by side
wm_wide = wm_df.pivot_table(
    index="comment_id",
    columns="perspective",
    values=[
        "hatespeech_0_prob",
        "hatespeech_1_prob",
        "hatespeech_2_prob",
        "annotation_count",
        "entropy"
    ],
    aggfunc="first"
)

wm_wide.columns = [
    f"{metric}_{perspective}"
    for metric, perspective in wm_wide.columns
]

wm_wide = wm_wide.reset_index()

# Only comments where BOTH women and men subgroup distributions exist
paired_wm = wm_wide.dropna(subset=[
    "hatespeech_0_prob_women",
    "hatespeech_1_prob_women",
    "hatespeech_2_prob_women",
    "hatespeech_0_prob_men",
    "hatespeech_1_prob_men",
    "hatespeech_2_prob_men",
]).copy()

print("Comments with both women and men subgroup distributions:", len(paired_wm))
paired_wm.head()

Comments with both women and men subgroup distributions: 112


,comment_id,annotation_count_men,annotation_count_women,entropy_men,entropy_women,hatespeech_0_prob_men,hatespeech_0_prob_women,hatespeech_1_prob_men,hatespeech_1_prob_women,hatespeech_2_prob_men,hatespeech_2_prob_women
55,726,2.0,2.0,1.0,1.0,0.5,0.5,0.0,0.0,0.5,0.5
95,1524,2.0,2.0,1.0,1.0,0.5,0.5,0.0,0.5,0.5,0.0
100,1615,2.0,2.0,1.0,1.0,0.5,0.5,0.0,0.0,0.5,0.5
146,2585,2.0,2.0,1.0,1.0,0.0,0.5,0.5,0.5,0.5,0.0
151,2652,2.0,2.0,1.0,1.0,0.5,0.5,0.0,0.0,0.5,0.5


In [33]:
# Calculate JS(women distribution, men distribution)
paired_wm["women_men_js"] = paired_wm.apply(
    lambda row: js_divergence(
        [
            row["hatespeech_0_prob_women"],
            row["hatespeech_1_prob_women"],
            row["hatespeech_2_prob_women"]
        ],
        [
            row["hatespeech_0_prob_men"],
            row["hatespeech_1_prob_men"],
            row["hatespeech_2_prob_men"]
        ]
    ),
    axis=1
)

paired_wm["women_entropy"] = paired_wm["entropy_women"]
paired_wm["men_entropy"] = paired_wm["entropy_men"]

paired_wm["entropy_gap"] = (
    paired_wm["women_entropy"] - paired_wm["men_entropy"]
).abs()

paired_wm["annotation_count_women"] = paired_wm["annotation_count_women"]
paired_wm["annotation_count_men"] = paired_wm["annotation_count_men"]

paired_wm[[
    "women_men_js",
    "entropy_gap",
    "women_entropy",
    "men_entropy",
    "annotation_count_women",
    "annotation_count_men"
]].describe()

,women_men_js,entropy_gap,women_entropy,men_entropy,annotation_count_women,annotation_count_men
count,112.000000,112.000000,112.000000,112.000000,112.000000,112.000000
mean,0.132149,0.062900,0.890597,0.885821,64.526786,47.160714
std,0.164989,0.109952,0.291790,0.307555,108.717334,79.512541
min,0.000000,0.000000,0.043328,0.070502,2.000000,2.000000
25%,0.000524,0.000000,0.916455,0.933999,2.000000,2.000000
50%,0.008522,0.000000,1.000000,1.000000,2.000000,2.000000
75%,0.346574,0.081704,1.000000,1.000000,118.500000,83.750000
max,0.412726,0.584963,1.459840,1.584963,423.000000,315.000000


In [34]:
# Interpret JS magnitude
paired_wm["js_group"] = pd.cut(
    paired_wm["women_men_js"],
    bins=[-0.001, 0.01, 0.05, 0.10, 0.20, 1.0],
    labels=[
        "near_identical_<=0.01",
        "small_0.01_0.05",
        "moderate_0.05_0.10",
        "large_0.10_0.20",
        "very_large_>0.20"
    ]
)

js_group_summary = (
    paired_wm["js_group"]
    .value_counts(normalize=False)
    .sort_index()
    .reset_index()
)

js_group_summary.columns = ["js_group", "comments"]
js_group_summary["percent"] = (
    js_group_summary["comments"] / len(paired_wm) * 100
).round(2)

js_group_summary

,js_group,comments,percent
0,near_identical_<=0.01,57,50.89
1,small_0.01_0.05,12,10.71
2,moderate_0.05_0.10,0,0.00
3,large_0.10_0.20,2,1.79
4,very_large_>0.20,41,36.61


In [35]:
# Examples where women and men differ most
top_gender_disagreement_examples = paired_wm.sort_values(
    "women_men_js",
    ascending=False
).head(20)

top_gender_disagreement_examples[[
    "comment_id",
    "women_men_js",
    "hatespeech_0_prob_women",
    "hatespeech_1_prob_women",
    "hatespeech_2_prob_women",
    "hatespeech_0_prob_men",
    "hatespeech_1_prob_men",
    "hatespeech_2_prob_men",
    "annotation_count_women",
    "annotation_count_men",
    "women_entropy",
    "men_entropy"
]]

,comment_id,women_men_js,hatespeech_0_prob_women,hatespeech_1_prob_women,hatespeech_2_prob_women,hatespeech_0_prob_men,hatespeech_1_prob_men,hatespeech_2_prob_men,annotation_count_women,annotation_count_men,women_entropy,men_entropy
909,15886,0.412726,0.333333,0.0,0.666667,0.5,0.5,0.0,3.0,2.0,0.918296,1.0
1955,34731,0.346574,0.000000,0.5,0.500000,0.5,0.0,0.5,2.0,2.0,1.000000,1.0
2425,43303,0.346574,0.500000,0.0,0.500000,0.0,0.5,0.5,2.0,2.0,1.000000,1.0
2400,43042,0.346574,0.500000,0.0,0.500000,0.0,0.5,0.5,2.0,2.0,1.000000,1.0
2390,42923,0.346574,0.500000,0.0,0.500000,0.5,0.5,0.0,2.0,2.0,1.000000,1.0
2240,40489,0.346574,0.000000,0.5,0.500000,0.5,0.0,0.5,2.0,2.0,1.000000,1.0
2120,38562,0.346574,0.500000,0.5,0.000000,0.5,0.0,0.5,2.0,2.0,1.000000,1.0
1728,30943,0.346574,0.500000,0.0,0.500000,0.0,0.5,0.5,2.0,2.0,1.000000,1.0
95,1524,0.346574,0.500000,0.5,0.000000,0.5,0.0,0.5,2.0,2.0,1.000000,1.0
1251,21497,0.346574,0.500000,0.0,0.500000,0.5,0.5,0.0,2.0,2.0,1.000000,1.0


In [36]:
# Add text back for inspection
text_lookup = (
    gender_df[["comment_id", "text_clean"]]
    .drop_duplicates("comment_id")
)

top_gender_disagreement_examples = top_gender_disagreement_examples.merge(
    text_lookup,
    on="comment_id",
    how="left"
)

top_gender_disagreement_examples[[
    "comment_id",
    "women_men_js",
    "annotation_count_women",
    "annotation_count_men",
    "hatespeech_0_prob_women",
    "hatespeech_1_prob_women",
    "hatespeech_2_prob_women",
    "hatespeech_0_prob_men",
    "hatespeech_1_prob_men",
    "hatespeech_2_prob_men",
    "text_clean"
]].head(20)

,comment_id,women_men_js,annotation_count_women,annotation_count_men,hatespeech_0_prob_women,hatespeech_1_prob_women,hatespeech_2_prob_women,hatespeech_0_prob_men,hatespeech_1_prob_men,hatespeech_2_prob_men,text_clean
0,15886,0.412726,3.0,2.0,0.333333,0.0,0.666667,0.5,0.5,0.0,"It's true, though. I don't understand how this..."
1,34731,0.346574,2.0,2.0,0.000000,0.5,0.500000,0.5,0.0,0.5,Bitches piss me off thot ass hoes
2,43303,0.346574,2.0,2.0,0.500000,0.0,0.500000,0.0,0.5,0.5,The raids began Friday in many area. It was wa...
3,43042,0.346574,2.0,2.0,0.500000,0.0,0.500000,0.0,0.5,0.5,ALSO START CHARGEING THESE COUNTRIES A DAILY F...
4,42923,0.346574,2.0,2.0,0.500000,0.0,0.500000,0.5,0.5,0.0,Loser woman
5,40489,0.346574,2.0,2.0,0.000000,0.5,0.500000,0.5,0.0,0.5,The only ones going to hell are those heathens...
6,38562,0.346574,2.0,2.0,0.500000,0.5,0.000000,0.5,0.0,0.5,Some girls really cause problems for themselve...
7,30943,0.346574,2.0,2.0,0.500000,0.0,0.500000,0.0,0.5,0.5,I hate dumb bitches
8,1524,0.346574,2.0,2.0,0.500000,0.5,0.000000,0.5,0.0,0.5,Cuntiest in like she possess the most cunt
9,21497,0.346574,2.0,2.0,0.500000,0.0,0.500000,0.5,0.5,0.0,"Oh, my. USA whore empire is playing with fire ..."


In [37]:
# Save gender divergence audit
import os

os.makedirs("../results/tables", exist_ok=True)

gender_divergence_path = "../results/tables/gender_women_men_distribution_divergence_audit.txt"

with open(gender_divergence_path, "w", encoding="utf-8") as f:
    f.write("GENDER SUBGROUP DISTRIBUTION DIVERGENCE AUDIT\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. PURPOSE\n")
    f.write("-" * 90 + "\n")
    f.write("This audit checks whether women and men subgroup-specific hatespeech distributions differ enough to justify gender-conditioned modelling.\n\n")

    f.write("2. PAIRED COMMENTS\n")
    f.write("-" * 90 + "\n")
    f.write(f"Comments with both women and men subgroup distributions: {len(paired_wm)}\n\n")

    f.write("3. JS DIVERGENCE SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(paired_wm[[
        "women_men_js",
        "entropy_gap",
        "women_entropy",
        "men_entropy",
        "annotation_count_women",
        "annotation_count_men"
    ]].describe().to_string())
    f.write("\n\n")

    f.write("4. JS GROUP SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(js_group_summary.to_string(index=False))
    f.write("\n\n")

    f.write("5. TOP GENDER-DIVERGENT EXAMPLES\n")
    f.write("-" * 90 + "\n")
    f.write(top_gender_disagreement_examples[[
        "comment_id",
        "women_men_js",
        "annotation_count_women",
        "annotation_count_men",
        "hatespeech_0_prob_women",
        "hatespeech_1_prob_women",
        "hatespeech_2_prob_women",
        "hatespeech_0_prob_men",
        "hatespeech_1_prob_men",
        "hatespeech_2_prob_men",
        "text_clean"
    ]].head(20).to_string(index=False))
    f.write("\n\n")

    

print("Saved:", gender_divergence_path)
print(open(gender_divergence_path, encoding="utf-8").read())

Saved: ../results/tables/gender_women_men_distribution_divergence_audit.txt
GENDER SUBGROUP DISTRIBUTION DIVERGENCE AUDIT

1. PURPOSE
------------------------------------------------------------------------------------------
This audit checks whether women and men subgroup-specific hatespeech distributions differ enough to justify gender-conditioned modelling.

2. PAIRED COMMENTS
------------------------------------------------------------------------------------------
Comments with both women and men subgroup distributions: 112

3. JS DIVERGENCE SUMMARY
------------------------------------------------------------------------------------------
       women_men_js  entropy_gap  women_entropy  men_entropy  annotation_count_women  annotation_count_men
count    112.000000   112.000000     112.000000   112.000000              112.000000            112.000000
mean       0.132149     0.062900       0.890597     0.885821               64.526786             47.160714
std        0.164989     0.1